<a href="https://colab.research.google.com/github/suwabe-j/practice-project/blob/main/chapter10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 〜〜〜〜csvデータ以外のデータフレーム化〜〜〜〜
# tsv（tabで文字が区切られており、認識しやすい）ファイルを読み込む（sepで区切り文字を明治）
import pandas as pd
df= pd.read_csv("bike.tsv", sep="\t")
df.head(3)
# 文字コードがshift-jisのcsvファイルを表に落としこむ（encodingで文字指定）
weather_df = pd.read_csv("weather.csv", encoding="shift-jis")
weather_df.head(3)
# JSON（データがリスト形式で記載されており、データ形式のデフォルトスタンダード）ファイル
temp = pd.read_json("temp.json")
temp.head(3)
# キーである数字がカラムになってしまっているので、カラムとインデックスを転置させる
temp.T
# 〜〜〜〜データの結合【内部結合 / 外部結合】〜〜〜〜
# weather.csv（カンマ区切り）とbike.tsv（tab区切り）の表をweather_idで内部結合させた表（df2）を作成する
# 内部結合＝共通の列に対して、同じ値の行同士を結合する方法
df2 = df.merge(weather_df, how='inner', on="weather_id")
# 内部結合のときは、行が順不同の場合があるので並び替えを行う
df2 = df2.sort_values(by="dteday")
df2.head(2)
# weather列の値ごとにcnt列の値をgroupby（集計）して、平均値を求める
df2.groupby("weather")["cnt"].mean()
# tempをdf2（weatherとbikeを結合した表）にくっつける（detday列の一部が抜け落ちているので外部結合にする）
# 外部結合：結合できなくても、せっかくの情報は残しておきたいときに使用する
df3 = df2.merge(temp.T , how="left", on="dteday")
df3.head(2)
# 〜〜〜〜欠損値の補完①【線形補完】〜〜〜〜
# df3には欠損値が含まれているので、その付近の折れ線グラフ（x=インデックス,y=atmep）を作成して補完されているか確認
# 間を線形補間するために、オブジェクト型ではできないので、float型に変更してからinterpolateする
df3["atemp"] = df3["atemp"].astype(float)
df3["atemp"] = df3['atemp'].interpolate()
# 線形補間されているか確認する
df3["atemp"].loc[695:705].plot(kind="line")
# 〜〜〜〜欠損の補完②【機械学習の利用】〜〜〜〜〜
# 機械学習モデルを作成して、そのモデルに欠損値を拡充してもらう
# まずはモデルを作成する（特徴量エンジニアリングや訓練データ云々は行わず）
iris_df = pd.read_csv("iris.csv")
# NaNになっている値（欠損値）を持つ行の削除
non_df = iris_df.dropna()
# モデルの作成（特徴量は３つ）
from sklearn.linear_model import LinearRegression
x = non_df.loc[ : ,"がく片幅":"花弁幅"]
t = non_df.loc[ : ,"がく片長さ"]
model = LinearRegression()
model.fit(x , t)
# 欠損値の予測をスタート
# 欠損値かどうかのTrueやFalseを集めたシリーズの情報を抜き取る
condition = iris_df["がく片長さ"].isnull()
# そのシリーズ情報を使用して、欠損値だけを集めた表を作成
non_data = iris_df[condition]
# 欠損値の補完に利用するモデルが使用する特徴量のみを抽出して、欠損値の予測値を作成
x = non_data.loc[ : ,"がく片幅":"花弁幅"]
pred = model.predict(x)
# 欠損値の補完
iris_df.loc[condition,"がく片長さ"] = pred
# 〜〜〜〜ハズレ値の処理【マハラノビス距離：分布の形を捉えたうえでの距離、箱ひげ図】〜〜〜〜
# convarianceは共分散の意味
from sklearn.covariance import MinCovDet
df4 = df3.loc[ : ,"atemp":"windspeed"]
df4 = df4.dropna()
# マハラノビス距離を作成するための共分散行列をmodelの中に情報を格納
# MinCovDetは密集している集団を探すために、ランダムに点の集合体を選んで密度を出しているのでrandom_stateを設定する
# spport_fractionは密集した点の集合体を構成する際に、全部の点のうち何％でそれを構成するかを設定する
mcd = MinCovDet(random_state = 0, support_fraction=0.7)
# 共分散行列をmodelの中に格納
mcd.fit(df4)
# マハラノビス距離の作成（Numpyの配列で生成）
distance = mcd.mahalanobis(df4)
# 箱ひげ図を作成し、どのマハラノビス距離を弾くのかを確認する（ハズレ値の確認）
distance = pd.Series(distance, index=df4.index)
distance.plot(kind="box")
# 値の幅が多くて箱ひげ図が潰れてしまっているので、視覚ではなく計算式で省く
# 基本統計量の表示
distance.describe()
# ハズレ値の計算
# 値が大きいハズレ値のしきい値（jogen）：第３四分位数＋1.5×IQR
# 値が小さいハズレ値のしきい値（kagen）：第１四分位数＋1.5×IQR
tmp = distance.describe()
iqr = tmp["75%"] - tmp["25%"]
jogen = tmp["75%"] + (1.5 * iqr)
kagen = tmp["25%"] + (1.5 * iqr)
# 条件検索
outliner = distance[(distance>jogen) | (distance<kagen)]
# outlinerのindexを吸収して、そのハズレ値を削除【完了】
df4_fin = df4.drop(outliner.index)